# Imports

In [2]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine

# Database connections and configuration

In [3]:
with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Database investigation and data loading

In [ ]:
# Probar la lectura de una tabla operacional real
try:
    query_prueba = "SELECT * FROM public.ciudad LIMIT 5;"
    df_ciudad = pd.read_sql_query(query_prueba, co_sa)
    print(" ¡Conexión operacional verificada con éxito!")
    display(df_ciudad)
except Exception as e:
    print(" Error al intentar leer de la base de datos:")
    print(e)
    print("\n Si sale error de autenticación o base de datos no encontrada, recuerda copiar 'config_fill.yml' a un archivo llamado 'config.yml', poner tus datos reales de Postgres y cambiar la ruta del código anterior a '../config.yml'.")

🚀 ¡Conexión operacional verificada con éxito!


,ciudad_id,nombre,departamento_id
0,6,BUGA,1
1,5,BOGOTA,2
2,4,PASTO,4
3,3,POPAYAN,3
4,2,PALMIRA,1


In [ ]:
# Inspeccionar las columnas de la tabla de servicios para buscar los campos geográficos
try:
    query_columnas = "SELECT * FROM public.mensajeria_servicio LIMIT 3;"
    df_servicio_muestra = pd.read_sql_query(query_columnas, co_sa)
    print(" Columnas encontradas en mensajeria_servicio:")
    print(df_servicio_muestra.columns.tolist())
except Exception as e:
    print(" Error al inspeccionar la tabla mensajeria_servicio:")
    print(e)

🚀 Columnas encontradas en mensajeria_servicio:
['id', 'descripcion', 'nombre_solicitante', 'fecha_solicitud', 'hora_solicitud', 'fecha_deseada', 'hora_deseada', 'nombre_recibe', 'telefono_recibe', 'descripcion_pago', 'ida_y_regreso', 'activo', 'novedades', 'cliente_id', 'destino_id', 'mensajero_id', 'origen_id', 'tipo_pago_id', 'tipo_servicio_id', 'tipo_vehiculo_id', 'usuario_id', 'prioridad', 'ciudad_destino_id', 'ciudad_origen_id', 'hora_visto_por_mensajero', 'visto_por_mensajero', 'descripcion_multiples_origenes', 'mensajero2_id', 'mensajero3_id', 'multiples_origenes', 'asignar_mensajero', 'es_prueba', 'descripcion_cancelado']


In [ ]:
# Inspeccionar las columnas de las tablas de direcciones detalladas
try:
    df_origenservicio = pd.read_sql_query("SELECT * FROM public.mensajeria_origenservicio LIMIT 3;", co_sa)
    df_destinoservicio = pd.read_sql_query("SELECT * FROM public.mensajeria_destinoservicio LIMIT 3;", co_sa)
    
    print(" Columnas encontradas en mensajeria_origenservicio (Origen):")
    print(df_origenservicio.columns.tolist())
    print("\ Columnas encontradas en mensajeria_destinoservicio (Destino):")
    print(df_destinoservicio.columns.tolist())
except Exception as e:
    print(" Error al inspeccionar las tablas de direcciones:")
    print(e)

📍 Columnas encontradas en mensajeria_origenservicio (Origen):
['id', 'direccion', 'cliente_id', 'ciudad_id']

🏁 Columnas encontradas en mensajeria_destinoservicio (Destino):
['id', 'direccion', 'cliente_id', 'ciudad_id']


# Extract


In [ ]:
# ==========================================
# 1. ETAPA: EXTRACT (Extracción de Datos)
# ==========================================
try:
    df_servicios = pd.read_sql_query("SELECT origen_id, destino_id FROM public.mensajeria_servicio;", co_sa)
    df_origenes = pd.read_sql_query("SELECT id, direccion, ciudad_id FROM public.mensajeria_origenservicio;", co_sa)
    df_destinos = pd.read_sql_query("SELECT id, direccion, ciudad_id FROM public.mensajeria_destinoservicio;", co_sa)
    df_ciudades = pd.read_sql_query("SELECT ciudad_id, nombre, departamento_id FROM public.ciudad;", co_sa)
    df_departamentos = pd.read_sql_query("SELECT departamento_id, nombre FROM public.departamento;", co_sa)
    
    print(" EXTRACT: Todas las tablas fuentes han sido leídas con éxito.")
    print(f"-> Servicios operacionales leídos: {len(df_servicios)}")
    
except Exception as e:
    print(" Error en la etapa de Extracción:")
    print(e)

✅ EXTRACT: Todas las tablas fuentes han sido leídas con éxito.
-> Servicios operacionales leídos: 28430


# Transform

In [8]:
# ==========================================
# 2. ETAPA: TRANSFORM (Transformación)
# ==========================================
from datetime import date

try:
    # A. Construir la información completa de ORIGEN
    origen_completo = pd.merge(df_origenes, df_ciudades, on='ciudad_id', how='left')
    origen_completo = pd.merge(origen_completo, df_departamentos, on='departamento_id', how='left')
    origen_completo = origen_completo.rename(columns={
        'id': 'origen_id',
        'direccion': 'Direccion_Origen',
        'nombre_x': 'Ciudad_Origen',
        'nombre_y': 'Departamento_Origen'
    })[['origen_id', 'Direccion_Origen', 'Ciudad_Origen', 'Departamento_Origen']]

    # B. Construir la información completa de DESTINO
    destino_completo = pd.merge(df_destinos, df_ciudades, on='ciudad_id', how='left')
    destino_completo = pd.merge(destino_completo, df_departamentos, on='departamento_id', how='left')
    destino_completo = destino_completo.rename(columns={
        'id': 'destino_id',
        'direccion': 'Direccion_Destino',
        'nombre_x': 'Ciudad_Destino',
        'nombre_y': 'Departamento_Destino'
    })[['destino_id', 'Direccion_Destino', 'Ciudad_Destino', 'Departamento_Destino']]

    # C. Unir Origen y Destino usando la tabla de servicios
    df_unificado = pd.merge(df_servicios, origen_completo, on='origen_id', how='left')
    dim_geografia = pd.merge(df_unificado, destino_completo, on='destino_id', how='left')

    # D. Limpieza Dimensional (Quedarse solo con columnas del modelo y remover duplicados)
    columnas_modelo = [
        'Ciudad_Origen', 'Departamento_Origen', 'Direccion_Origen',
        'Ciudad_Destino', 'Departamento_Destino', 'Direccion_Destino'
    ]
    dim_geografia = dim_geografia[columnas_modelo].drop_duplicates().reset_index(drop=True)
    
    # E. Columna de auditoría
    dim_geografia['saved'] = date.today()

    print("✅ TRANSFORM: Modelado dimensional completado.")
    print(f"-> Rutas geográficas únicas listas para cargar: {len(dim_geografia)}")
    display(dim_geografia.head(5))

except Exception as e:
    print("❌ Error en la etapa de Transformación:")
    print(e)

✅ TRANSFORM: Modelado dimensional completado.
-> Rutas geográficas únicas listas para cargar: 38


,Ciudad_Origen,Departamento_Origen,Direccion_Origen,Ciudad_Destino,Departamento_Destino,Direccion_Destino,saved
0,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
1,ACOPI YUMBO,VALLE DEL CAUCA,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
2,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,CENCAR YUMBO,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
3,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,PALMIRA,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
4,PALMIRA,VALLE DEL CAUCA,Los angeles distrito Latino,PALMIRA,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22


# Load 

In [9]:
# ==========================================
# 3. ETAPA: LOAD (Carga en Data Warehouse)
# ==========================================
try:
    # Guardamos en la base de datos conectada en 'etl_conn'
    # 'if_exists="append"' añade filas nuevas. 
    # 'index=False' evita que Pandas guarde la columna oculta del índice.
    dim_geografia.to_sql('dim_geografia', con=etl_conn, if_exists='append', index=False)
    
    print("🚀 ¡LOAD EXITOSO!")
    print("La dimensión 'dim_geografia' ha sido cargada en la base de datos de destino con éxito.")
    
except Exception as e:
    print("❌ Error en la etapa de Carga (Load):")
    print(e)
    print("\n👉 Nota: Si sale un error de que la tabla ya existe y tiene restricciones de llave primaria,")
    print("puedes intentar cambiar 'append' por 'replace' de forma temporal para pruebas.")

🚀 ¡LOAD EXITOSO!
La dimensión 'dim_geografia' ha sido cargada en la base de datos de destino con éxito.


In [10]:
# Validar si existen direcciones diferentes en las tablas fuente originales
print("Direcciones únicas en Origen operacional:", df_origenes['direccion'].nunique())
print("Direcciones únicas en Destino operacional:", df_destinos['direccion'].nunique())

# Mostrar una muestra directa de la tabla fuente de orígenes
display(df_origenes[['id', 'direccion']].head(5))

Direcciones únicas en Origen operacional: 1
Direcciones únicas en Destino operacional: 1


,id,direccion
0,94,Los angeles distrito Latino
1,95,Los angeles distrito Latino
2,96,Los angeles distrito Latino
3,55,Los angeles distrito Latino
4,79,Los angeles distrito Latino


# Viusalization dim_geografia

In [11]:
# ==========================================================
# VERIFICACIÓN: Consultar los datos reales en el Data Warehouse
# ==========================================================
try:
    # Hacemos una consulta SELECT a la base analítica usando el motor 'etl_conn'
    query_olap = "SELECT * FROM public.dim_geografia;"
    df_resultado_olap = pd.read_sql_query(query_olap, etl_conn)
    
    print(f"✅ ¡Verificación Exitosa! Se encontraron {len(df_resultado_olap)} registros guardados en la OLAP.")
    
    # Mostrar la tabla completa o las primeras filas
    display(df_resultado_olap)

except Exception as e:
    print("❌ Error al intentar leer la tabla desde el Data Warehouse:")
    print(e)

✅ ¡Verificación Exitosa! Se encontraron 38 registros guardados en la OLAP.


,Ciudad_Origen,Departamento_Origen,Direccion_Origen,Ciudad_Destino,Departamento_Destino,Direccion_Destino,saved
0,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
1,ACOPI YUMBO,VALLE DEL CAUCA,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
2,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,CENCAR YUMBO,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
3,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,PALMIRA,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
4,PALMIRA,VALLE DEL CAUCA,Los angeles distrito Latino,PALMIRA,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
5,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,ACOPI YUMBO,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
6,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,JAMUNDI,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
7,PALMIRA,VALLE DEL CAUCA,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
8,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,CANDELARIA,VALLE DEL CAUCA,Los angeles distrito Latino,2026-06-22
9,CALI,VALLE DEL CAUCA,Los angeles distrito Latino,POPAYAN,CAUCA,Los angeles distrito Latino,2026-06-22
